# Experiment 04 — Response Planning

```
Language comprehension
      ↓
Meaning + memory + emotion + social context
      ↓
Response planning   <- this notebook
      ↓
Word and sentence construction
      ↓
Motor commands
```

Stage 03 handed off a fused "situational" vector. Before anything gets said, a
decision has to be made about *what kind of response fits* — answer the question
directly? Ask for clarification first? Lead with empathy? Give an instruction? This
stage is that decision: a classifier from situation to response strategy.

Standalone as always: we synthesize stand-in situational vectors with a known
generating rule, rather than importing stage 03's model.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0);

## A toy rule with three hidden factors

A real 16-dim fused vector from stage 03 wouldn't have individually-interpretable
dimensions, but we need *some* structure to generate labels from. So: pick three
scalar factors — **clarity** (how clear the input was), **distress** (how much
emotional weight it's carrying), and **directness_needed** (does this call for
instructions vs. conversation) — write each into one dimension of an otherwise
random 16-dim vector, and use them to define the four planning strategies:

- distress is high -> **empathize**, regardless of anything else
- otherwise, clarity is low -> **ask_clarifying_question**
- otherwise, directness_needed is high -> **give_instruction**
- otherwise -> **answer_directly**

In [2]:
plans = ["answer_directly", "ask_clarifying_question", "empathize", "give_instruction"]


def plan_rule(clarity, distress, directness_needed):
    if distress > 0.6:
        return "empathize"
    if clarity < 0.4:
        return "ask_clarifying_question"
    if directness_needed > 0.6:
        return "give_instruction"
    return "answer_directly"


def make_situational_vector(clarity, distress, directness_needed, noise_scale=0.3):
    vec = torch.randn(16) * noise_scale
    vec[0] = clarity * 2 - 1
    vec[1] = distress * 2 - 1
    vec[2] = directness_needed * 2 - 1
    return vec


def make_dataset(n):
    vectors, labels = [], []
    for _ in range(n):
        clarity, distress, directness = torch.rand(3).tolist()
        vectors.append(make_situational_vector(clarity, distress, directness))
        labels.append(plans.index(plan_rule(clarity, distress, directness)))
    return torch.stack(vectors), torch.tensor(labels)


train_X, train_y = make_dataset(200)
test_X, test_y = make_dataset(50)
print("train label counts:", {p: (train_y == i).sum().item() for i, p in enumerate(plans)})

train label counts: {'answer_directly': 39, 'ask_clarifying_question': 56, 'empathize': 83, 'give_instruction': 22}


## Architecture: classify, then look up a plan embedding

A small MLP maps the 16-dim situational vector to one of 4 strategies. We also keep
a learned embedding per strategy — that embedding, not just the discrete label, is
what stage 05 would actually condition on (a discrete "which of 4 plans" signal is
hard for a generator to use directly; a vector is easier).

In [3]:
class ResponsePlanner(nn.Module):
    def __init__(self, situational_dim=16, hidden_dim=32, n_plans=4, plan_embed_dim=8):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(situational_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_plans),
        )
        self.plan_embed = nn.Embedding(n_plans, plan_embed_dim)

    def forward(self, situational_vec):
        logits = self.classifier(situational_vec)
        return logits

    def embed_plan(self, plan_idx):
        return self.plan_embed(plan_idx)


model = ResponsePlanner()
logits = model(train_X)
print("logits shape:", logits.shape)

logits shape: torch.Size([200, 4])


## Training

Cross-entropy, Adam, held-out test set generated by the same rule.

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []
for step in range(400):
    logits = model(train_X)
    loss = F.cross_entropy(logits, train_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

with torch.no_grad():
    train_acc = (model(train_X).argmax(dim=1) == train_y).float().mean().item()
    test_acc = (model(test_X).argmax(dim=1) == test_y).float().mean().item()

print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"train accuracy: {train_acc:.2%}   test accuracy: {test_acc:.2%}")

loss: 1.343 -> 0.001
train accuracy: 100.00%   test accuracy: 88.00%


## Planning for four situations

One hand-built situational vector per rule branch, showing the predicted plan and
the plan embedding stage 05 would actually receive (first 4 of its 8 dims, to keep
the printout readable).

In [5]:
situations = [
    ("clear question, low distress, low directness", 0.9, 0.1, 0.1),
    ("unclear/ambiguous input", 0.1, 0.1, 0.5),
    ("high distress, regardless of clarity", 0.8, 0.9, 0.2),
    ("clear request that needs instructions", 0.9, 0.1, 0.9),
]

for description, clarity, distress, directness in situations:
    vec = make_situational_vector(clarity, distress, directness).unsqueeze(0)
    with torch.no_grad():
        logits = model(vec)
        plan_idx = logits.argmax(dim=1)
        plan_vec = model.embed_plan(plan_idx).squeeze(0)
    predicted = plans[plan_idx.item()]
    print(f"{description:45} -> {predicted:24} embed[:4]={plan_vec[:4].tolist()}")

clear question, low distress, low directness  -> answer_directly          embed[:4]=[0.3156360983848572, 1.1309763193130493, -1.0543268918991089, -0.9046222567558289]
unclear/ambiguous input                       -> ask_clarifying_question  embed[:4]=[-1.6071979999542236, -0.612321138381958, -0.3140837848186493, 1.6209397315979004]
high distress, regardless of clarity          -> empathize                embed[:4]=[-2.454732894897461, 0.8105900883674622, 0.6350752711296082, -0.12520520389080048]
clear request that needs instructions         -> give_instruction         embed[:4]=[-2.281170129776001, 0.10958308726549149, 1.4338431358337402, -0.24155814945697784]


## What this hands off (conceptually)

An 8-dim `plan` embedding, one per chosen strategy. Stage 05 (`word and sentence
construction`) would condition its word-by-word generation on a vector like this —
the plan says *what kind* of thing to say; the next stage decides the actual
words.